# Step 21.5 — Context cropping：保留输出历史，限制模型窗口

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/96_step21_5_context_cropping.ipynb notebooks/work/96_step21_5_context_cropping_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。当前母版是**材料初稿 0/2**，不是学习完成证明。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

生成长度可以大于训练时的 block_size，但一次 forward 不能越过学到的位置表容量。最简单的处理是保留完整输出 history，每轮只把其最后 W 个 token 送入模型。裁剪的是模型上下文，不是用户最后看到的生成文本。

这不是长上下文能力升级。被裁掉的内容不再影响下一步预测；固定窗口里的 learned absolute positions 也会重新从零编号。原模型并不因此拥有超出训练长度的位置理解。

## 2. 先备数学与 Python / MLX 语义

定义完整长度 Lᵣ=L₀+r，实际输入长度 Tᵣ=min(Lᵣ,W)。Python `ids[:, -W:]` 从末尾取最多 W 项；若当前长度小于 W，不会补齐，不会产生负长度。最后分布为 p(xₙ₊₁ | xₘ,…,xₙ)，m=max(1,n−W+1)，是有限上下文条件分布。

无 KV cache 时，每轮都重新计算整个窗口的 embedding、投影、attention 和 MLP。Attention score 主项每轮约 O(W²d) 每头；N 个 decode 步约 O(NW²d)，忽略其他项与增长阶段。这是算术规模估算，不是运行速度测量。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

W=3，完整 history=[A,B,C,A]，forward 只接收 [B,C,A]，位置编号 [0,1,2]。若下一 token 为 B，history=[A,B,C,A,B]，下一窗口为 [C,A,B]。两轮重叠 token 仍被重算。

纸笔比较：若 W=4，prompt 长 2，生成前三次 forward 长度依次是 2、3、4；第 4 次仍是 4，不是 5。输出列表长度则继续增长。

## 4. Shape tracing 与接口契约

history (1,Lᵣ) → context (1,min(Lᵣ,W)) → logits (1,min(Lᵣ,W),V) → last (1,V) → next (1,1) → history (1,Lᵣ+1)。返回完整 history 与每次 context 长度记录，用形状证据分离“输出长度”和“模型容量”。本例仅 greedy，无 EOS/batch/cache。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

# 人工定义转移分数；不是语料训练结果。
TOY_VOCAB = ("A", "B", "C", "_")
TRANSITION = mx.array([[0., 3., 1., -1.],
                       [0., 0., 3., -1.],
                       [3., 0., 0., -1.],
                       [3., 1., 0., -1.]], dtype=mx.float32)

def toy_logits(ids):
    # 输入 int32 (B,T)，合法 ID 0..3；输出 (B,T,4)。
    # 这个模型只看当前 token，故意不模拟真实 Transformer。
    return TRANSITION[ids]

def check_prompt(prompt, vocab_size):
    if not isinstance(prompt, (list, tuple)) or not prompt:
        raise ValueError("prompt must be a nonempty list/tuple of IDs")
    if any(type(t) is not int or not 0 <= t < vocab_size for t in prompt):
        raise ValueError("prompt contains an invalid token ID")

def last_logits(logits_fn, ids, vocab_size):
    logits = logits_fn(ids)
    if logits.shape != (*ids.shape, vocab_size):
        raise ValueError("callback must return (B,T,V) logits")
    last = logits[:, -1, :].astype(mx.float32)
    mx.eval(last)
    if not bool(mx.all(mx.isfinite(last)).item()):
        raise ValueError("model logits must be finite")
    return last

def generate_cropped(logits_fn, prompt, max_new_tokens, *, vocab_size, block_size):
    check_prompt(prompt, vocab_size)
    if type(block_size) is not int or block_size <= 0:
        raise ValueError("block_size must be a positive integer")
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens must be a nonnegative integer")
    history = mx.array([prompt], dtype=mx.int32)
    lengths = []
    for _ in range(max_new_tokens):
        context = history[:, -block_size:]
        lengths.append(context.shape[1])
        last = last_logits(logits_fn, context, vocab_size)
        token = mx.argmax(last, axis=-1).astype(mx.int32)[:, None]
        history = mx.concatenate([history, token], axis=1)
        mx.eval(history)
    return history, lengths

## 5. 短实验计划

先预测 shape 与趋势，再在未来工作副本中运行下一个单元。这里没有运行记录。若未来结果和推导不同，先检查输入、axis、dtype、mask、随机 key，再讨论数值误差；不要把一次随机输出当成分布证明。

In [ ]:
history, context_lengths = generate_cropped(
    toy_logits, [0, 1], 5, vocab_size=4, block_size=3)
mx.eval(history)
print("forward context lengths:", context_lengths)
print("full output:", history.tolist())

## 6. 常见误区与未来衔接

KV cache（键值缓存）为每层保存已经算过的 K、V，decode 只算新 token 的 Q/K/V，再让新 Q 关注缓存。缓存张量通常为 (B,Hkv,T,d)，每层 K/V 两份。它减少重复计算，同时增加驻留内存，不是免费优化。

不能给当前 learned-position 窗口简单拼一个 cache 就宣称等价：窗口滑动时旧 token 的绝对位置改变，深层 hidden states 的可见前缀也改变，旧 K/V 可能已经不对应“从头重算窗口”的定义。RoPE offset、窗口策略、cache 生命周期、prefill/decode mask 必须共同设计。本节只预告，不实现 KV cache、不修改 MiniGPT。

真实模型需要 W≤max_seq_len，并核对训练 block_size 与位置表容量。Toy 只依赖当前 token，不能用它证明真实 Transformer 裁剪前后输出等价。

## 7. 练习（不附答案）

1. W=4、prompt 长 6、生成 3 个 token 时，列出每轮窗口索引和返回长度。
2. 用文字说明为什么裁剪 history 本身会丢失用户需要的完整输出。
3. 画两层 KV cache 形状，指出 learned-position 滑窗时哪些假设可能失效。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 / python/src/ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)

接口说明以固定版本源码为准。本次只有文本与源代码静态检查，没有运行通过声明。能够复述概念与完成练习后再进入下一节；材料覆盖范围不代表读者已经掌握。